# Stock Market Analysis - Implementation Notebook

This notebook starts the implementation of a stock market analysis workflow using Python, Pandas, NumPy, Matplotlib, and Seaborn.

Dataset used:
- `.sixth/Global_Stock_Market_Indices_2000_2026.csv`

The notebook covers loading, cleaning, metrics, visualization, and trend/volatility analysis.

## 1. Import Libraries and Configure the Environment

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style and display options
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

ROOT = Path.cwd().parent
DATA_PATH = ROOT / ".sixth" / "Global_Stock_Market_Indices_2000_2026.csv"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Load Historical Stock Data

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=["Date"])

print("Raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
print("Date range:", df_raw["Date"].min(), "to", df_raw["Date"].max())
print("Unique tickers:", df_raw["Ticker"].nunique())

df_raw.head()

## 3. Clean and Prepare the Dataset

In [ ]:
df = df_raw.copy()

numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove duplicates and rows missing core values
df = df.drop_duplicates(subset=["Date", "Ticker"])
df = df.dropna(subset=["Date", "Open", "High", "Low", "Close", "Ticker", "Index_Name"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Missing values (top):")
print(df.isna().sum().sort_values(ascending=False).head(10))

ticker_summary = (
    df.groupby("Ticker")
    .agg(rows=("Date", "count"), start_date=("Date", "min"), end_date=("Date", "max"), zero_volume=("Volume", lambda s: int((s == 0).sum())))
    .sort_values("rows", ascending=False)
)
ticker_summary.head(15)

## 4. Compute Core Market Metrics

In [ ]:
df_metrics = df.copy()

df_metrics["Daily_Return"] = df_metrics.groupby("Ticker")["Close"].pct_change()
df_metrics["Pct_Change"] = df_metrics["Daily_Return"] * 100
df_metrics["Price_Range"] = df_metrics["High"] - df_metrics["Low"]
df_metrics["Cumulative_Return"] = (1 + df_metrics["Daily_Return"].fillna(0)).groupby(df_metrics["Ticker"]).cumprod() - 1

summary_stats = (
    df_metrics.groupby("Ticker")
    .agg(
        mean_daily_return=("Daily_Return", "mean"),
        std_daily_return=("Daily_Return", "std"),
        mean_price_range=("Price_Range", "mean"),
        latest_cumulative_return=("Cumulative_Return", "last"),
    )
    .sort_values("latest_cumulative_return", ascending=False)
)

summary_stats.head(15)

## 5. Visualize Price, Volume, and Returns

In [ ]:
# Normalized price chart for all tickers
fig, ax = plt.subplots(figsize=(14, 7))
for ticker, chunk in df_metrics.groupby("Ticker"):
    normalized = chunk["Close"] / chunk["Close"].iloc[0]
    ax.plot(chunk["Date"], normalized, linewidth=0.9, label=ticker)
ax.set_title("Normalized Price Performance (All Tickers)")
ax.set_xlabel("Date")
ax.set_ylabel("Normalized Close")
ax.legend(loc="upper left", ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb_normalized_prices.png", dpi=150)
plt.show()

# Price and volume for a selected ticker
ticker = sorted(df_metrics["Ticker"].unique())[0]
ticker_df = df_metrics[df_metrics["Ticker"] == ticker]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(ticker_df["Date"], ticker_df["Close"], color="tab:blue")
ax1.set_title(f"Close Price - {ticker}")
ax1.set_ylabel("Close")

ax2.bar(ticker_df["Date"], ticker_df["Volume"], color="tab:gray", alpha=0.5)
ax2.set_title(f"Volume - {ticker}")
ax2.set_ylabel("Volume")
ax2.set_xlabel("Date")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"nb_price_volume_{ticker.replace('^','')}.png", dpi=150)
plt.show()

# Return distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_metrics["Daily_Return"].dropna(), bins=80, kde=True, ax=ax)
ax.set_title("Distribution of Daily Returns (All Tickers)")
ax.set_xlabel("Daily Return")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb_return_distribution.png", dpi=150)
plt.show()

## 6. Analyze Moving Averages and Volatility

In [ ]:
# Moving averages and rolling volatility
short_window = 20
long_window = 50
vol_window = 20

for w in (short_window, long_window):
    df_metrics[f"SMA_{w}"] = df_metrics.groupby("Ticker")["Close"].transform(lambda s: s.rolling(w).mean())

df_metrics[f"Volatility_{vol_window}"] = (
    df_metrics.groupby("Ticker")["Daily_Return"].transform(lambda s: s.rolling(vol_window).std() * np.sqrt(252))
)

selected = sorted(df_metrics["Ticker"].unique())[0]
selected_df = df_metrics[df_metrics["Ticker"] == selected].copy()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(selected_df["Date"], selected_df["Close"], label="Close", linewidth=1.0)
ax.plot(selected_df["Date"], selected_df[f"SMA_{short_window}"], label=f"SMA {short_window}", linewidth=1.0)
ax.plot(selected_df["Date"], selected_df[f"SMA_{long_window}"], label=f"SMA {long_window}", linewidth=1.0)
ax.set_title(f"Moving Averages - {selected}")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"nb_moving_averages_{selected.replace('^','')}.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(selected_df["Date"], selected_df[f"Volatility_{vol_window}"], color="tab:red")
ax.set_title(f"Rolling Annualized Volatility ({vol_window}d) - {selected}")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"nb_volatility_{selected.replace('^','')}.png", dpi=150)
plt.show()

# Correlation heatmap across tickers
corr = df_metrics.pivot_table(index="Date", columns="Ticker", values="Daily_Return").corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Daily Return Correlation Across Tickers")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb_correlation_heatmap.png", dpi=150)
plt.show()

# RSI and MACD on selected ticker
def compute_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

selected_df["RSI_14"] = compute_rsi(selected_df["Close"], 14)
ema_fast = selected_df["Close"].ewm(span=12, adjust=False).mean()
ema_slow = selected_df["Close"].ewm(span=26, adjust=False).mean()
selected_df["MACD"] = ema_fast - ema_slow
selected_df["MACD_Signal"] = selected_df["MACD"].ewm(span=9, adjust=False).mean()
selected_df["MACD_Hist"] = selected_df["MACD"] - selected_df["MACD_Signal"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(selected_df["Date"], selected_df["RSI_14"], color="tab:purple")
ax1.axhline(70, linestyle="--", color="red", linewidth=0.8)
ax1.axhline(30, linestyle="--", color="green", linewidth=0.8)
ax1.set_title(f"RSI (14) - {selected}")
ax1.set_ylabel("RSI")

ax2.plot(selected_df["Date"], selected_df["MACD"], label="MACD")
ax2.plot(selected_df["Date"], selected_df["MACD_Signal"], label="Signal")
ax2.bar(selected_df["Date"], selected_df["MACD_Hist"], alpha=0.3, label="Hist")
ax2.set_title("MACD")
ax2.set_xlabel("Date")
ax2.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"nb_rsi_macd_{selected.replace('^','')}.png", dpi=150)
plt.show()

# Combined dashboard
fig = plt.figure(figsize=(16, 10))

ax1 = fig.add_subplot(2, 2, 1)
for t, chunk in df_metrics.groupby("Ticker"):
    ax1.plot(chunk["Date"], chunk["Close"] / chunk["Close"].iloc[0], linewidth=0.8)
ax1.set_title("Normalized Prices (All Tickers)")
ax1.set_xlabel("Date")
ax1.set_ylabel("Normalized")

ax2 = fig.add_subplot(2, 2, 2)
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax2, cbar=True)
ax2.set_title("Correlation Heatmap")

ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(selected_df["Date"], selected_df["Close"], label="Close")
ax3.plot(selected_df["Date"], selected_df[f"SMA_{short_window}"], label=f"SMA {short_window}")
ax3.plot(selected_df["Date"], selected_df[f"SMA_{long_window}"], label=f"SMA {long_window}")
ax3.set_title(f"Moving Averages - {selected}")
ax3.legend()

ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(selected_df["Date"], selected_df[f"Volatility_{vol_window}"], color="tab:red")
ax4.set_title(f"Volatility ({vol_window}d) - {selected}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb_dashboard_combined.png", dpi=150)
plt.show()

# Save processed notebook dataset
df_metrics.to_csv(OUTPUT_DIR / "nb_features_dataset.csv", index=False)
summary_stats.to_csv(OUTPUT_DIR / "nb_summary_stats.csv")
print("Saved notebook outputs to:", OUTPUT_DIR)